# Лаба 05. RAG в цикле вопросов-ответов

Тема семестрового проекта — **проверка Go-кода по код-стайлу команды**. Агент из лаб 2–3 (наставник с памятью
фактов, памятью диалога, развилками и инструментом `check_code`) теперь отвечает на вопросы о стиле Go
**по базе знаний** — руководствам, а не по памяти модели.

Как устроено:

- **База знаний** (`kb/`): командный код-стайл S1–S6 (по-русски, свой текст) и три открытых руководства —
  Go Code Review Comments, Effective Go, Uber Go Style Guide (по-английски; снимок скачан `fetch_kb.py`,
  ссылки и лицензии — `kb/sources.json`).
- **Индексация** (`kb.py`): `MarkdownHeaderTextSplitter` режет по заголовкам, `RecursiveCharacterTextSplitter` —
  длинные разделы до 1200 символов; к тексту чанка добавляется «Источник. Раздел». Эмбеддинги — **bge-m3**
  (мультиязычная: вопрос по-русски находит английский раздел) во втором llama-server с `--embeddings`,
  через `OpenAIEmbeddings`. Хранилище — `InMemoryVectorStore`, индекс сохраняется в файл и пересчитывается,
  только если поменялись документы.
- **RAG в цикле вопросов-ответов** (`fact_agent.py`): после `TurnParser` идёт шаг поиска. Для объяснения правила
  и для вопроса найденные фрагменты попадают в промпт, модель ссылается на них номерами [1], [2], список
  источников дописывает код. Близость лучшего фрагмента — новая развилка: выше порога — вопрос по Go,
  ответ по базе; ниже — не по теме (порог калибруется на вопросах с известным ответом и вопросах не по теме).

Модели: Qwen2.5-3B-Instruct Q4_K_M (чат) и bge-m3 Q8_0 (эмбеддинги), обе в llama-server.

```bash
pip install -r requirements.txt
llama-server -m qwen2.5-3b-instruct-q4_k_m.gguf -c 8192 --port 8080
llama-server -m bge-m3-Q8_0.gguf --embeddings --pooling cls -c 8192 -b 8192 -ub 8192 --port 8081
```

In [1]:
import os

# чат-модель и модель эмбеддингов — два OpenAI-совместимых сервера (llama-server локально или на Kaggle)
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # Qwen2.5-3B-Instruct
os.environ.setdefault("LLM_MODEL", "local")
os.environ.setdefault("EMB_BASE_URL", "http://127.0.0.1:8081/v1")  # bge-m3, llama-server --embeddings
os.environ.setdefault("EMB_MODEL", "bge-m3")

KB_DIR = "kb"                # руководства по стилю Go (Markdown)
INDEX_FILE = "kb_index.json"  # векторный индекс на диске
MEMORY_FILE = "memory.json"
USER_ID = "artur"

Код базы знаний и агента сохраняется в модули, чтобы после перезапуска ядра их можно было импортировать.

In [2]:
%%writefile kb.py
"""База знаний для RAG: руководства по стилю Go → чанки → эмбеддинги → векторное хранилище.

Источники (kb/*.md): командный код-стайл S1–S6 (по-русски), Go Code Review Comments, Effective Go,
Uber Go Style Guide (по-английски; ссылки и лицензии — kb/sources.json). Вопросы пользователя — по-русски,
поэтому модель эмбеддингов мультиязычная: bge-m3 в llama-server с --embeddings (OpenAI-совместимый API).
"""

import hashlib
import json
import os
import re
import time
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4")]
CHUNK_SIZE, CHUNK_OVERLAP = 1200, 150


def make_embeddings() -> OpenAIEmbeddings:
    return OpenAIEmbeddings(
        base_url=os.environ.get("EMB_BASE_URL", "http://127.0.0.1:8081/v1"),
        api_key=os.environ.get("EMB_API_KEY", "none"),
        model=os.environ.get("EMB_MODEL", "bge-m3"),
        check_embedding_ctx_length=False,  # иначе режет текст токенизатором tiktoken под модели OpenAI
        chunk_size=32,  # текстов в одном запросе
    )


def load_chunks(kb_dir: str | Path = "kb") -> list[Document]:
    """Markdown → разделы по заголовкам → куски до CHUNK_SIZE символов.

    В текст чанка добавляется «Источник. Раздел»: заголовок раздела часто и есть ответ на вопрос
    («Receiver Names»), а сплиттер убирает его из тела.
    """
    kb_dir = Path(kb_dir)
    sources = json.loads((kb_dir / "sources.json").read_text(encoding="utf-8"))
    by_headers = MarkdownHeaderTextSplitter(HEADERS, strip_headers=True)
    by_size = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = []
    for name, src in sources.items():
        text = (kb_dir / name).read_text(encoding="utf-8")
        for part in by_size.split_documents(by_headers.split_text(text)):
            lines = [l for l in part.page_content.splitlines() if l.strip()]
            # оглавления (списки ссылок) и обрывки без смысла в индекс не берём
            if len(part.page_content) < 100 or sum(l.lstrip().startswith("- [") for l in lines) > len(lines) / 2:
                continue
            section = " > ".join(part.metadata[h] for h in ("h2", "h3", "h4") if h in part.metadata)
            chunks.append(Document(
                page_content=f"{src['title']}. {section}\n{part.page_content}",
                metadata={"source": name, "title": src["title"], "section": section, "url": src.get("page", "")},
            ))
    return chunks


class KnowledgeBase:
    """Векторное хранилище чанков. Индекс сохраняется в файл и пересчитывается, только если поменялись чанки."""

    def __init__(self, store: InMemoryVectorStore, threshold: float = 0.5, meta: Path | None = None):
        self.store, self.threshold, self.meta = store, threshold, meta

    @classmethod
    def build_or_load(cls, kb_dir: str | Path = "kb", index: str | Path = "kb_index.json", log=None, **kwargs):
        log = log or print
        chunks, embeddings, index = load_chunks(kb_dir), make_embeddings(), Path(index)
        digest = hashlib.sha256(json.dumps([c.page_content for c in chunks]).encode()).hexdigest()[:16]
        meta = index.with_suffix(".meta.json")
        saved = json.loads(meta.read_text()) if meta.exists() else {}
        if index.exists() and saved.get("digest") == digest:
            store = InMemoryVectorStore.load(str(index), embeddings)
            log(f"индекс прочитан из `{index}`: {len(store.store)} чанков, эмбеддинги не пересчитывались"
                + (f"; порог {saved['threshold']}" if "threshold" in saved else ""))
            if "threshold" in saved:
                kwargs.setdefault("threshold", saved["threshold"])
        else:
            t = time.time()
            store = InMemoryVectorStore(embeddings)
            store.add_documents(chunks)
            store.dump(str(index))
            meta.write_text(json.dumps({"digest": digest, "chunks": len(chunks)}))
            log(f"индекс построен: {len(chunks)} чанков за {time.time() - t:.0f} с, сохранён в `{index}`")
        return cls(store, meta=meta, **kwargs)

    def search(self, query: str, k: int = 4) -> list[tuple[Document, float]]:
        """Косинусная близость запроса и чанков, по убыванию."""
        return self.store.similarity_search_with_score(query, k=k)

    def calibrate(self) -> dict:
        """Порог «вопрос по Go / не по теме»: середина между худшим вопросом из EVAL и лучшим из OFF_TOPIC.

        Сохраняется рядом с индексом — после перезапуска ядра не нужно калибровать снова.
        """
        inside = {q: self.search(q, 1)[0][1] for q, _ in EVAL}
        outside = {q: self.search(q, 1)[0][1] for q in OFF_TOPIC}
        self.threshold = round((min(inside.values()) + max(outside.values())) / 2, 3)
        if self.meta:
            self.meta.write_text(json.dumps({**json.loads(self.meta.read_text()), "threshold": self.threshold}))
        return {"inside": inside, "outside": outside, "threshold": self.threshold}


def cite(doc: Document) -> str:
    return f"{doc.metadata['title']} — {doc.metadata['section'] or 'введение'}"


def group_hits(hits: list[tuple[Document, float]]) -> list[list[Document]]:
    """Чанки одного раздела — один фрагмент: иначе в источниках раздел повторяется («Error Wrapping» дважды)."""
    groups: dict[tuple, list[Document]] = {}
    for doc, _ in hits:
        groups.setdefault((doc.metadata["title"], doc.metadata["section"]), []).append(doc)
    return list(groups.values())


def format_context(hits: list[tuple[Document, float]]) -> str:
    return "\n\n".join(f"[{i}] {cite(docs[0])}\n" + "\n".join(d.page_content.split("\n", 1)[-1] for d in docs)
                       for i, docs in enumerate(group_hits(hits), 1))


def format_sources(hits: list[tuple[Document, float]]) -> str:
    return "Источники:\n" + "\n".join(f"[{i}] {cite(docs[0])} ({docs[0].metadata['url']})"
                                      for i, docs in enumerate(group_hits(hits), 1))


def strip_sources(answer: str) -> str:
    """Свой список источников модель дописывает по образцу прошлых ответов и путает — оставляем только код."""
    return re.split(r"\n\s*\**Источники\**:?", answer)[0].strip()


# --- классическая RAG-цепочка LCEL: для сравнения «модель без базы / с базой» ---

PLAIN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Ты — эксперт по стилю кода на Go. Отвечай по-русски, кратко: 3–5 предложений."),
    ("human", "{question}"),
])
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — эксперт по стилю кода на Go. Отвечай по-русски, кратко: 3–5 предложений. Отвечай только по "
     "фрагментам руководств ниже (они могут быть на английском), ссылайся на них номерами [1], [2]. "
     "Если в фрагментах ответа нет — так и скажи. Список источников не пиши — его добавят.\n\nФрагменты:\n{context}"),
    ("human", "{question}"),
])


def make_plain_chain(llm):
    return PLAIN_PROMPT | llm | StrOutputParser()


def make_rag_chain(kb: KnowledgeBase, llm, k: int = 4):
    """question → поиск → контекст в промпт → ответ модели + список источников (его пишет код, не модель)."""
    retrieve = RunnableLambda(lambda q: kb.search(q, k))
    return (
        RunnablePassthrough.assign(hits=lambda x: retrieve.invoke(x["question"]))
        | RunnablePassthrough.assign(answer=(lambda x: {"question": x["question"],
                                                         "context": format_context(x["hits"])})
                                     | RAG_PROMPT | llm | StrOutputParser())
        | RunnableLambda(lambda x: f"{strip_sources(x['answer'])}\n\n{format_sources(x['hits'])}")
    )


# --- оценка поиска: вопрос → какие разделы считаются верным ответом (источник, кусок названия раздела) ---

EVAL = [
    ("Как называть ресивер метода?", [("code_review_comments.md", "Receiver Names")]),
    ("Когда метод должен принимать указатель, а когда значение?",
     [("code_review_comments.md", "Receiver Type"), ("effective_go.md", "Pointers vs. Values"),
      ("uber_style.md", "Receivers and Interfaces")]),
    ("С какой буквы начинать текст ошибки и ставить ли точку в конце?", [("code_review_comments.md", "Error Strings")]),
    ("Как добавить контекст к ошибке, чтобы errors.Is продолжал работать?",
     [("uber_style.md", "Error Wrapping"), ("team_style.md", "S5")]),
    ("Можно ли использовать panic для обработки обычных ошибок?",
     [("code_review_comments.md", "Don't Panic"), ("uber_style.md", "Don't Panic"), ("team_style.md", "S3")]),
    ("Где объявлять интерфейс: рядом с реализацией или там, где он используется?",
     [("code_review_comments.md", "Interfaces")]),
    ("Как писать аббревиатуры вроде URL и ID в именах?",
     [("code_review_comments.md", "Initialisms"), ("team_style.md", "S1")]),
    ("Можно ли хранить context.Context в поле структуры?",
     [("code_review_comments.md", "Contexts"), ("team_style.md", "S4")]),
    ("С чего должен начинаться комментарий к экспортируемой функции?",
     [("code_review_comments.md", "Doc Comments"), ("effective_go.md", "Commentary"), ("team_style.md", "S6")]),
    ("Как правильно называть пакеты?",
     [("effective_go.md", "Package names"), ("code_review_comments.md", "Package Names"),
      ("uber_style.md", "Package Names")]),
    ("Как объявить пустой срез: через var или через литерал?",
     [("code_review_comments.md", "Declaring Empty Slices"), ("uber_style.md", "nil is a valid slice")]),
    ("Стоит ли давать имена возвращаемым значениям функции?", [("code_review_comments.md", "Named Result Parameters")]),
    ("Что вернуть клиенту, если в HTTP-обработчике упал запрос к базе?", [("team_style.md", "S3")]),
    ("Можно ли не проверять ошибку, которую вернул Close?", [("team_style.md", "S2"), ("uber_style.md", "Defer")]),
]
OFF_TOPIC = [
    "Какая сегодня погода?",
    "Посоветуй рецепт борща.",
    "Кто выиграл чемпионат мира по футболу?",
    "Сколько стоит биткоин?",
]


def evaluate(kb: KnowledgeBase, k: int = 4) -> list[dict]:
    rows = []
    for question, relevant in EVAL:
        hits = kb.search(question, k)
        ranks = [i for i, (doc, _) in enumerate(hits, 1)
                 if any(doc.metadata["source"] == src and part.lower() in doc.metadata["section"].lower()
                        for src, part in relevant)]
        rows.append({"question": question, "rank": ranks[0] if ranks else None, "top1": cite(hits[0][0]),
                     "score": round(hits[0][1], 3)})
    return rows


Writing kb.py


In [3]:
%%writefile fact_agent.py
"""Чат-наставник по Go-код-стайлу команды с долгосрочной памятью фактов в файле.

Память — не история сообщений, а набор фактов о пользователе (имя, уровень, стек, цели...).
Факты извлекаются отдельным LLM-вызовом со structured output и хранятся в memory.json по user_id.
История сообщений живёт только в RAM текущего сеанса и после перезапуска пропадает; в промпт она
попадает не целиком: окно последних сообщений (buffer) или окно плюс резюме вытесненных (summary).

Когда факты собраны — режим наставника: классификатор реплики → TurnParser (свой Runnable: разбирает
параметры и решает, куда идти) → RunnableBranch: объяснить правило, проверить код инструментом check_code,
квиз по уровню знаний (прогресс — в memory.json), упростить объяснение, если пользователь недоволен.

RAG (лаба 5): перед ответом на вопрос или объяснением правила агент ищет фрагменты руководств по стилю Go
в базе знаний (kb.py) и отвечает по ним со ссылками [1], [2]; список источников дописывает код. Близость
лучшего фрагмента — ещё одна развилка: выше порога — вопрос по Go, ответ по базе; ниже — не по теме.
"""

import json
import os
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Literal, Optional

from IPython.display import Markdown, display
from langchain_classic.memory.chat_memory import BaseChatMemory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core._api import LangChainDeprecationWarning
from pydantic import BaseModel, Field, create_model

from kb import cite, format_context, format_sources, strip_sources

# предупреждения об устаревших классах мешаются с выводом чата
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

STYLE_GUIDE = """\
S1. Имена — MixedCaps/camelCase, без подчёркиваний. Аббревиатуры целиком: userID, не userId.
S2. Ошибки не игнорировать: ни `_ = f()`, ни вызов без проверки возвращаемой ошибки. `defer x.Close()` допускается.
S3. В HTTP-обработчиках не использовать panic — вернуть 500 через http.Error.
S4. Функции, которые ходят в БД или сеть, принимают ctx context.Context первым параметром.
    HTTP-обработчики берут контекст из r.Context() — отдельный параметр ctx им не нужен.
S5. Ошибки, возвращаемые наверх, оборачиваются: fmt.Errorf("контекст: %w", err).
S6. У экспортируемых функций и методов есть doc-комментарий, начинающийся с имени."""
RULES = {rule: " ".join(text.split())
         for rule, text in re.findall(r"^(S\d)\. (.+?)(?=^S\d\.|\Z)", STYLE_GUIDE, re.S | re.M)}
RULE_WORDS = {  # о каком правиле речь, если номер не назван
    "S1": ("имен", "camel", "подчёрк", "подчерк", "аббревиатур"),
    "S2": ("игнор", "не проверя"),
    "S3": ("panic", "паник"),
    "S4": ("context", "ctx", "контекст"),
    "S5": ("оборач", "errorf", "%w", "ошибк"),
    "S6": ("коммент",),
}
RULE_NUMBER = r"(?<!\w)[SsСс]\s?([1-6])(?!\d)"  # S3, s3, С3 (кириллица)


def find_rule(text: str) -> Optional[str]:
    if m := re.search(RULE_NUMBER, text):
        return f"S{m.group(1)}"
    low = text.lower()
    return next((rule for rule, words in RULE_WORDS.items() if any(w in low for w in words)), None)


def lint_go(code: str) -> list[tuple[int, str, str]]:
    """Простой линтер по правилам S1–S6 на регулярках: (строка, правило, что не так)."""
    found = []
    lines = code.splitlines()
    handler = "http.ResponseWriter" in code
    for n, line in enumerate(lines, 1):
        s = line.strip()
        if s.startswith("//"):
            continue
        names = re.findall(r"\b(?:func\s+(?:\([^)]*\)\s*)?|var\s+|const\s+)([A-Za-z_]\w*)", s)
        if m := re.match(r"(?:if\s+|for\s+)?([\w\s,]+?)\s*:=", s):
            names += [x.strip() for x in m.group(1).split(",")]
        for name in names:
            if "_" in name.strip("_"):
                found.append((n, "S1", f"имя `{name}` с подчёркиванием"))
            elif re.search(r"(Id|Url|Http|Json|Sql)(?=[A-Z]|$)", name):
                found.append((n, "S1", f"аббревиатура в `{name}` — целиком заглавными"))
        if re.search(r"(?:^|[\s(])_\s*(?:,\s*_\s*)?=", s) or re.search(r",\s*_\s*:?=", s):
            found.append((n, "S2", "ошибка присвоена `_`"))
        elif re.fullmatch(r"[\w.]+(?:\([^)]*\))?\.(?:Encode|Decode|Write|Exec|Scan)\(.*\)", s):
            found.append((n, "S2", "результат-ошибка вызова не проверяется"))
        if handler and re.search(r"\bpanic\(", s):
            found.append((n, "S3", "panic в HTTP-обработчике"))
        if m := re.search(r"\.(Query|QueryRow|Exec)\((?!\))", s):  # у url.Query() аргументов нет
            found.append((n, "S4", f"запрос к БД без контекста: нужен {m.group(1)}Context(ctx, ...)"))
        if re.fullmatch(r"return\s+(?:[^,]+,\s*)*err", s):
            found.append((n, "S5", "ошибка возвращается без обёртки fmt.Errorf(\"...: %w\", err)"))
        elif "fmt.Errorf(" in s and "err" in s and "%w" not in s:
            found.append((n, "S5", "в fmt.Errorf ошибка через %v, а не %w"))
        if m := re.match(r"func\s+(?:\([^)]*\)\s*)?([A-Z]\w*)", s):
            prev = lines[n - 2].strip() if n > 1 else ""
            if not prev.startswith(f"// {m.group(1)} "):
                found.append((n, "S6", f"у экспортируемой `{m.group(1)}` нет doc-комментария, начинающегося с имени"))
    return found


@tool
def check_code(code: str) -> str:
    """Проверяет Go-код по правилам команды S1–S6 и возвращает нарушения с номерами строк."""
    return "\n".join(f"строка {n}: {rule} — {what}" for n, rule, what in lint_go(code)) or "нарушений не найдено"


# Квиз: в каждом фрагменте одно нарушение, ответ — номер правила. Проверяет ответ код, объяснение — why:
# пересказывая ответ, 3B-модель противоречила вердикту («Неверно: S5. Верно: S2…»)
QUIZ = [
    {"id": "j1", "level": "junior", "rule": "S1",
     "code": "func load_config(path string) (*Config, error) {",
     "why": "`load_config` — имя с подчёркиванием, по правилам `loadConfig`."},
    {"id": "j2", "level": "junior", "rule": "S2",
     "code": "data, _ := os.ReadFile(path)",
     "why": "ошибка из `os.ReadFile` присвоена `_`: нужно `data, err := os.ReadFile(path)` и проверить `err`."},
    {"id": "j3", "level": "junior", "rule": "S3",
     "code": "func ordersHandler(w http.ResponseWriter, r *http.Request) {\n"
             "    orders, err := repo.List(r.Context())\n    if err != nil {\n        panic(err)\n    }",
     "why": "panic в HTTP-обработчике роняет обработку запроса; нужно `http.Error(w, \"internal error\", http.StatusInternalServerError)` и `return`."},
    {"id": "m1", "level": "middle", "rule": "S4",
     "code": "// FindUser ищет пользователя по id.\nfunc (r *Repo) FindUser(id int64) (*User, error) {\n"
             "    row := r.db.QueryRow(\"SELECT name FROM users WHERE id = $1\", id)",
     "why": "`FindUser` ходит в БД, но не принимает контекст: нужно `FindUser(ctx context.Context, id int64)` и `QueryRowContext(ctx, ...)`."},
    {"id": "m2", "level": "middle", "rule": "S5",
     "code": "if err := row.Scan(&u.Name); err != nil {\n    return nil, err\n}",
     "why": "ошибка проверена, но уходит наверх без контекста: нужно `return nil, fmt.Errorf(\"scan user: %w\", err)`. S2 — это когда ошибку не проверяют вовсе, а здесь её проверяют."},
    {"id": "m3", "level": "middle", "rule": "S6",
     "code": "func ParseOrder(b []byte) (*Order, error) {",
     "why": "у экспортируемой `ParseOrder` нет doc-комментария; нужен `// ParseOrder разбирает заказ из JSON.`"},
    {"id": "s1", "level": "senior", "rule": "S1",
     "code": "// GetUserId возвращает id текущего пользователя.\n"
             "func (s *Service) GetUserId(ctx context.Context) (int64, error) {",
     "why": "аббревиатура пишется целиком: `GetUserID`, а не `GetUserId`."},
    {"id": "s2", "level": "senior", "rule": "S5",
     "code": "return nil, fmt.Errorf(\"load order %d: %v\", id, err)",
     "why": "с `%v` ошибка превращается в текст и `errors.Is`/`errors.As` перестают её видеть; нужен `%w`."},
    {"id": "s3", "level": "senior", "rule": "S6",
     "code": "// возвращает заказы клиента\nfunc ListOrders(ctx context.Context, clientID int64) ([]Order, error) {",
     "why": "комментарий есть, но не начинается с имени функции: нужно `// ListOrders возвращает заказы клиента.`"},
    {"id": "s4", "level": "senior", "rule": "S2",
     "code": "func statsHandler(w http.ResponseWriter, r *http.Request) {\n"
             "    json.NewEncoder(w).Encode(collectStats())\n}",
     "why": "`Encode` возвращает ошибку, а её не проверяют: `if err := json.NewEncoder(w).Encode(...); err != nil { http.Error(...) }`."},
]
LEVELS = ["junior", "middle", "senior"]
DEPTH = {
    "junior": "объясняй простыми словами, без лишних терминов; обязательно короткий пример кода «плохо / хорошо»",
    "middle": "объясни правило и зачем оно нужно, дай короткий пример «плохо / хорошо»",
    "senior": "коротко: суть правила, неочевидные случаи и исключения; пример — только если он показывает тонкость",
}


class UserFacts(BaseModel):
    """Факты о разработчике. Поле заполняется, только если пользователь сам это сказал."""

    name: Optional[str] = Field(None, description="Имя пользователя")
    # Literal превращается в enum схемы: грамматика llama-server не даст записать сюда «год»
    go_level: Optional[Literal["junior", "middle", "senior"]] = Field(None, description="Уровень в Go")
    go_experience: Optional[str] = Field(None, description="Сколько времени пишет на Go: «1 год», «полгода»")
    stack: Optional[str] = Field(None, description="Технологии кроме Go: базы данных, инструменты, ОС")
    project: Optional[str] = Field(None, description="Над каким проектом или сервисом работает")
    weak_spots: Optional[str] = Field(None, description="Что даётся трудно в Go или в код-стайле")
    goal: Optional[str] = Field(None, description="Чего хочет добиться")


# порядок опроса и дословные вопросы: 3B-модель, получив «узнай уровень», спрашивает что-то своё
QUESTIONS = {
    "name": "Как тебя зовут?",
    "go_level": "Какой у тебя уровень в Go: junior, middle или senior?",
    "go_experience": "Сколько ты уже пишешь на Go?",
    "stack": "С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?",
    "project": "Над каким проектом ты сейчас работаешь?",
    "weak_spots": "Что тебе сейчас даётся труднее всего в Go или в код-стайле?",
    "goal": "Чего ты хочешь добиться?",
}
LABELS = {
    "name": "Имя", "go_level": "Уровень", "go_experience": "Опыт в Go", "stack": "Стек",
    "project": "Проект", "weak_spots": "Слабые места", "goal": "Цель",
}
INCIDENTAL = ("name", "go_level", "go_experience")  # можно назвать, отвечая на другой вопрос
EMPTY = {"", "null", "none", "нет", "неизвестно", "не указано", "n/a"}


def make_llm(temperature: float = 0.3) -> ChatOpenAI:
    """Любой OpenAI-совместимый сервер: llama-server локально/на Kaggle или OpenRouter."""
    return ChatOpenAI(
        base_url=os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1"),
        api_key=os.environ.get("LLM_API_KEY", "none"),
        model=os.environ.get("LLM_MODEL", "local"),
        temperature=temperature,
        max_tokens=400,
        # langchain-openai шлёт лимит как max_completion_tokens, а llama-server b6981 его не знает —
        # без max_tokens в теле запроса модель, зациклившись, генерировала до конца контекста
        extra_body={"max_tokens": 400},
    )


class LogCell:
    """Служебный вывод в собственной ячейке ноутбука, а не в чате.

    Ячейка, в которой создан LogCell, держит display_id; все записи обновляют её вывод.
    """

    def __init__(self, title: str = "Служебный лог"):
        self.title = title
        self.lines: list[str] = []
        self.handle = display(Markdown(self._render()), display_id=True)

    def __call__(self, text: str):
        self.lines.append(f"- `{datetime.now():%H:%M:%S}` {text}")
        self.handle.update(Markdown(self._render()))

    def _render(self) -> str:
        return f"**{self.title}**\n\n" + ("\n".join(self.lines) or "_пока пусто_")


def format_facts(facts: dict) -> str:
    if not facts:
        return "пока ничего не известно"
    return "\n".join(f"- {LABELS[k]}: {v}" for k, v in facts.items() if k in LABELS)


EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты извлекаешь факты о пользователе из его последнего сообщения и возвращаешь JSON по схеме.\n"
     "Заполняй поле, только если пользователь сам прямо сказал это в своём сообщении. "
     "Всё, о чём он не сказал, — null. Не придумывай, не копируй слова из вопроса ассистента. "
     "Пиши значения коротко, по-русски."),
    ("human", "Ассистент спрашивал про поле «{field}».\nВопрос ассистента: {question}\n"
              "Сообщение пользователя: {message}"),
])


LEVEL_WORDS = {"junior": ("junior", "джун"), "middle": ("middle", "мидл"), "senior": ("senior", "сеньор", "синьор")}


def accept_fact(key: str, value: str, message: str, asked: str) -> bool:
    """Проверка, что факт действительно есть в реплике: маленькая модель заполняет поля «на всякий случай»."""
    text = message.lower()
    if key == "name":
        return value.lower() in text and (asked == "name" or "зовут" in text or re.search(r"\bя\s", text))
    if key == "go_level":
        return any(word in text for word in LEVEL_WORDS[value])
    if key == "go_experience":
        unit = r"\d|год|лет|месяц|полгода|недел"
        return bool(re.search(unit, value.lower())) and bool(re.search(unit, text))
    # стек, проект, слабые места, цель — только в ответ на свой вопрос
    return key == asked


class FileFactMemory(BaseChatMemory):
    """Долгосрочная память фактов в JSON-файле: {user_id: {"facts": {...}, "progress": {...}, "updated_at": ...}}.

    load_memory_variables() — отдаёт известные факты и список недостающих;
    save_context() — извлекает факты из реплики пользователя отдельным LLM-вызовом и пишет файл;
    progress — уровень знаний по квизу и ошибки по правилам (меняется в режиме наставника, save_progress()).
    """

    path: Path
    user_id: str
    llm: Any = None  # модель для извлечения фактов
    log: Any = None
    facts: dict = Field(default_factory=dict)
    progress: dict = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        profile = self._read_all().get(self.user_id)
        self.facts = dict(profile["facts"]) if profile else {}
        self.progress = dict(profile.get("progress", {})) if profile else {}
        if self.log:
            if not self.path.exists():
                self.log(f"`{self.path}` нет — память пустая")
            elif profile:
                self.log(f"прочитан `{self.path}`, профиль `{self.user_id}`: "
                         f"`{json.dumps(self.facts, ensure_ascii=False)}`")
                if self.progress:
                    self.log(f"прогресс: `{json.dumps(self.progress, ensure_ascii=False)}`")
            else:
                self.log(f"в `{self.path}` нет профиля `{self.user_id}` — память пустая")

    @property
    def memory_variables(self) -> list[str]:
        return ["facts", "missing"]

    def missing(self) -> list[str]:
        return [k for k in QUESTIONS if k not in self.facts]

    def load_memory_variables(self, inputs: dict[str, Any]) -> dict[str, Any]:
        return {"facts": format_facts(self.facts), "missing": self.missing()}

    def save_context(self, inputs: dict[str, Any], outputs: dict[str, str]) -> None:
        """inputs: {"input": реплика, "question": вопрос ассистента, "field": о каком факте он был}.

        Историю сообщений не сохраняем — только факты, прошедшие проверку.
        """
        if not self.missing():
            if self.log:
                self.log("все факты известны — извлечение не нужно")
            return
        message, field = inputs["input"], inputs.get("field") or "—"
        # В схеме — поле, о котором спросили, и недостающие короткие факты, которые называют попутно.
        # Иначе 3B-модель раскладывает перечисление по полям: stack="PostgreSQL", project="Docker"...
        fields = [k for k in self.missing() if k == field or k in INCIDENTAL]
        spec = {k: (UserFacts.model_fields[k].annotation, UserFacts.model_fields[k]) for k in fields}
        if field in fields and field not in INCIDENTAL:
            # Необязательное поле модель охотно оставляет null («Биллинг: считаем платежи...» → project=null),
            # поэтому спрошенное поле — обязательная строка с выходом «нет».
            spec[field] = (str, Field(description=UserFacts.model_fields[field].description
                                      + "; если пользователь об этом не сказал — «нет»"))
        schema = create_model("UserFacts", **spec)
        extractor = EXTRACT_PROMPT | self.llm.with_structured_output(schema, method="json_schema")
        result = extractor.invoke({"message": message, "question": inputs.get("question") or "—", "field": field})
        raw = {k: v.strip() for k, v in result.model_dump().items()
               if isinstance(v, str) and v.strip().lower() not in EMPTY}
        accepted, rejected = {}, {}
        for key, value in raw.items():
            if key in self.facts:
                rejected[key] = "уже известно"
            elif not accept_fact(key, value, message, field):
                rejected[key] = value
            else:
                accepted[key] = value
        if self.log:
            self.log(f"модель извлекла: `{json.dumps(raw, ensure_ascii=False)}`")
            if rejected:
                self.log(f"отброшено проверкой: `{json.dumps(rejected, ensure_ascii=False)}`")
        if accepted:
            self.facts.update(accepted)
            self._write()
            if self.log:
                self.log(f"записано в `{self.path}`: {', '.join(accepted)}")

    def level(self) -> str:
        """Уровень для сложности тем и вопросов: из квиза, а до первого квиза — названный пользователем."""
        return self.progress.get("level") or self.facts.get("go_level") or "junior"

    def save_progress(self, **changes) -> None:
        self.progress.update(changes)
        self._write()
        if self.log:
            self.log(f"прогресс записан в `{self.path}`: `{json.dumps(self.progress, ensure_ascii=False)}`")

    def clear(self) -> None:
        self.facts, self.progress = {}, {}
        self._write()

    def _read_all(self) -> dict:
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text(encoding="utf-8") or "{}")

    def _write(self) -> None:
        data = self._read_all()
        data[self.user_id] = {"facts": self.facts, "updated_at": datetime.now().isoformat(timespec="seconds")}
        if self.progress:
            data[self.user_id]["progress"] = self.progress
        self.path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")




SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Перескажи реплики разговора наставника по Go с разработчиком одним предложением, которое начинается "
     "со слов «Пользователь»: что он сообщил или о чём спросил. Сохрани имена людей и названия. "
     "Объяснения наставника и код не пересказывай. Только то, что есть в репликах, по-русски."),
    ("human", "{lines}"),
])


class DialogHistory(BaseChatMessageHistory):
    """История сообщений сеанса (только RAM); в промпт попадает не вся.

    buffer  — последние window сообщений, остальное просто отбрасывается;
    summary — те же window сообщений плюс резюме вытесненных: выпавшие из окна реплики отдельный LLM-вызов
              пересказывает одним-двумя предложениями, пункт добавляется в резюме (оно идёт в системный промпт).
              Пункты копятся, а не переписываются: переписывая резюме целиком, 3B-модель теряла старое.
    """

    def __init__(self, mode: str = "summary", window: int = 6, llm: Any = None, log: Any = None):
        assert mode in ("buffer", "summary")
        self.mode, self.window, self.llm = mode, window, llm
        self.log = log or (lambda text: None)
        self.full: list[BaseMessage] = []
        self.points: list[str] = []  # резюме — по пункту на свёрнутый кусок
        self.summarized = 0  # сколько первых сообщений уже свёрнуто в резюме

    @property
    def messages(self) -> list[BaseMessage]:
        # token_counter=len — считаем сообщения, а не токены
        return trim_messages(self.full, strategy="last", token_counter=len, max_tokens=self.window,
                             start_on="human")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.full.extend(messages)
        dropped = len(self.full) - len(self.messages)
        if self.mode == "summary" and dropped > self.summarized:
            chunk = self.full[self.summarized:dropped]
            # одно приветствие не сворачиваем: из него модель сочиняла целый диалог
            if any(m.type == "human" for m in chunk):
                # ответы наставника обрезаем: иначе 3B-модель копирует их в резюме вместо пересказа
                lines = "\n".join(f"Пользователь: {m.content}" if m.type == "human" else
                                   f"Наставник: {m.content[:150]}…" for m in chunk)
                lines = re.sub(r"```.*?(```|$)", "[код]", lines, flags=re.S)
                point = (SUMMARY_PROMPT | self.llm).invoke({"lines": lines}).content.strip()
                self.points.append(re.sub(r"^(новое )?резюме:\s*", "", point, flags=re.I))
                self.log(f"резюме: свёрнуто сообщений {len(chunk)} → «{self.points[-1]}»")
            self.summarized = dropped

    def prompt_summary(self) -> str:
        if self.mode == "summary" and self.points:
            return "Краткое содержание более раннего разговора:\n" + "\n".join(f"- {p}" for p in self.points) + "\n\n"
        return ""

    def clear(self) -> None:
        self.full, self.points, self.summarized = [], [], 0


CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — наставник по код-стайлу Go в команде разработки. Отвечай только по-русски, без английских "
     "и китайских слов (кроме кода), на «ты», кратко: 2–4 предложения.\n\n"
     "Что ты знаешь о пользователе (долгосрочная память):\n{facts}\n\n"
     "{summary}Задача на этот ход: {task}"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])


class Route(BaseModel):
    """Класс реплики пользователя в режиме наставника."""

    intent: Literal["explain_rule", "review_code", "quiz", "other"] = Field(
        description="explain_rule — просит объяснить правило или тему код-стайла, показать пример; "
                    "review_code — прислал код на Go или просит проверить код; "
                    "quiz — просит проверить его знания, задать вопрос, устроить тест; "
                    "other — всё остальное")
    rule: Literal["S1", "S2", "S3", "S4", "S5", "S6", "нет"] = Field(
        description="О каком правиле речь; «нет», если ни о каком")
    # поле mood («недоволен») убрано: в первом прогоне модель ставила его почти на каждую реплику


ROUTE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Определи, чего хочет пользователь наставника по код-стайлу Go, и верни JSON по схеме.\n"
               f"Правила команды:\n{STYLE_GUIDE}"),
    ("human", "{input}"),
])

NEGATIVE = re.compile(r"не понял|не понимаю|непонятн|не ясно|запутал|слишком сложно|бред|ерунд|\bне то\b"
                      r"|вс[её] равно|опять", re.I)
QUIZ_WORDS = re.compile(r"квиз|тест|проверь меня|проверь мои знания|задай.{0,10}вопрос|ещё вопрос|еще вопрос", re.I)
MORE_WORDS = re.compile(r"^(ещё|еще|дальше|давай|следующий)\b", re.I)
EXPLAIN_WORDS = re.compile(r"объясн|расскаж|покажи|пример|что такое|зачем", re.I)
CODE_BLOCK = re.compile(r"```(?:go)?\s*\n(.*?)```", re.S)


def similar(a: str, b: str) -> bool:
    """Повтор той же просьбы: больше половины слов совпадает."""
    wa, wb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(wa) >= 3 and len(wa & wb) / len(wa | wb) >= 0.5


class TurnParser(Runnable[dict, dict]):
    """Свой компонент цепочки: разбирает реплику и ответ классификатора в параметры хода.

    На входе {"input", "route": Route, ...}, на выходе тот же словарь плюс intent, rule, code, reason.
    3B-модель ошибается в классе («Объясни правило S3» → review_code), поэтому явные признаки важнее её
    ответа: блок кода → проверка кода (без кода проверять нечего), ожидается ответ на вопрос квиза → проверка
    ответа, «не понял» или повтор просьбы → упростить, номер правила → объяснить.
    """

    def __init__(self, agent: "FactAgent"):
        self.agent = agent

    def invoke(self, input: dict, config=None, **kwargs) -> dict:
        agent, text, route = self.agent, input["input"], input["route"]
        intent, reasons = route.intent, []
        code = m.group(1) if (m := CODE_BLOCK.search(text)) else (
            text if "\n" in text and re.search(r"\bfunc\b|:=|\bif err\b", text) else "")
        if code:
            intent = "review_code"
        elif agent.pending_quiz and len(text.split()) <= 4 and not EXPLAIN_WORDS.search(text) and (
                re.search(r"(?<!\w)[SsСс]?\s?[1-6](?!\d)", text) or re.search(r"не знаю", text, re.I)):
            # ответ на квиз — короткий («S2», «наверное, 2»); «Объясни правило S5» — не ответ
            intent = "quiz_answer"
        elif QUIZ_WORDS.search(text) or (agent.last_intent in ("quiz", "quiz_answer") and MORE_WORDS.search(text)):
            intent = "quiz"
        elif intent == "quiz":
            intent = "other"  # квиз — только по просьбе: модель отнесла к квизу вопрос про имя получателя
        elif intent == "review_code" or (intent == "other" and re.search(RULE_NUMBER, text)):
            intent = "explain_rule"
        if intent == "explain_rule" and not (find_rule(text) or EXPLAIN_WORDS.search(text)):
            intent = "other"  # «Кто у нас делает ревью?» — не про правило, хоть модель и назвала S1

        negative = bool(NEGATIVE.search(text))
        repeated = similar(text, agent.last_user)
        if negative:
            reasons.append("в реплике недовольство или непонимание")
        if repeated:
            reasons.append("повторяет прошлую просьбу")
        if reasons and intent in ("explain_rule", "other"):
            intent = "simplify"
        if agent.kb and intent == "explain_rule" and not (re.search(RULE_NUMBER, text) or EXPLAIN_WORDS.search(text)):
            # «Можно ли хранить ctx в поле структуры?» — вопрос, а не просьба объяснить S4 целиком: решит поиск
            intent = "other"

        weak = find_rule(agent.memory.facts.get("weak_spots", ""))
        if intent == "simplify":  # упрощаем то, что объясняли, а не правило, названное в жалобе
            rule = agent.last_rule or find_rule(text) or weak
        else:
            rule = find_rule(text) or (route.rule if route.rule != "нет" else None)
            if intent == "other" and rule and EXPLAIN_WORDS.search(text):
                intent = "explain_rule"
            if intent == "explain_rule" and not rule:
                rule = weak or agent.last_rule or "S1"
        agent.frustration = agent.frustration + 1 if intent == "simplify" else 0
        if intent != "simplify":
            agent.quiz_context = None
        agent.last_intent = intent
        if intent in ("explain_rule", "simplify") and rule:
            agent.last_rule = rule
        agent.log(f"классификатор: `{route.model_dump_json()}` → парсер: intent=`{intent}`, rule=`{rule}`"
                  + (f", причина: {'; '.join(reasons)}" if reasons else ""))
        return {**input, "intent": intent, "rule": rule, "code": code, "reason": "; ".join(reasons)}


class FactAgent:
    def __init__(self, user_id: str, path: str | Path = "memory.json", log: LogCell | None = None,
                 dialog: str = "summary", window: int = 6, kb: Any = None, k: int = 4):
        self.log = log or (lambda text: None)
        self.memory = FileFactMemory(
            path=Path(path), user_id=user_id, log=log, llm=make_llm(0))
        self.history = DialogHistory(dialog, window, llm=make_llm(0), log=self.log)
        self.chain = CHAT_PROMPT | make_llm(0.3)
        self.chat_chain = RunnableWithMessageHistory(
            self.chain, self._history, input_messages_key="input", history_messages_key="history")
        self.classifier = ROUTE_PROMPT | make_llm(0).with_structured_output(Route, method="json_schema")
        mentor = (
            RunnablePassthrough.assign(route=self.classifier)
            | TurnParser(self)
            | RunnableLambda(self._retrieve)
            | RunnableBranch(
                (lambda x: x["intent"] == "review_code", RunnableLambda(self._review)),
                (lambda x: x["intent"] == "quiz_answer", RunnableLambda(self._grade)),
                (lambda x: x["intent"] == "quiz", RunnableLambda(self._quiz)),
                (lambda x: x["intent"] == "simplify", RunnableLambda(self._simplify)),
                (lambda x: x["intent"] == "question", RunnableLambda(self._answer)),
                (lambda x: x["intent"] == "explain_rule", RunnableLambda(self._explain)),
                RunnableLambda(self._other),
            )
        )
        self.mentor_chain = RunnableWithMessageHistory(
            mentor, self._history, input_messages_key="input", history_messages_key="history")
        self.last_question = ""
        self.asked = ""  # о каком факте был последний вопрос ассистента
        # состояние режима наставника (RAM сеанса)
        self.pending_quiz: dict | None = None
        self.last_intent = self.last_rule = self.last_user = ""
        self.frustration = 0  # сколько ходов подряд пользователь недоволен
        self.quiz_context: dict | None = None  # неверный ответ на квиз — о чём упрощать
        self.kb, self.k = kb, k
        self.log(f"память диалога: `{dialog}`, окно {window} сообщений; "
                 + (f"база знаний: {len(kb.store.store)} чанков, порог {kb.threshold}" if kb else "без базы знаний"))

    def _history(self, session_id: str) -> DialogHistory:
        return self.history

    def _task(self, start: bool) -> str:
        missing = self.memory.missing()
        if missing:
            ask = f"затем задай дословно этот вопрос и больше ничего не спрашивай: «{QUESTIONS[missing[0]]}»"
            if start and not self.memory.facts:
                return "поздоровайся и одной фразой представься; " + ask
            if start:
                return "пользователь вернулся: поприветствуй его по имени и одной фразой напомни, что уже знаешь; " + ask
            return "одной фразой отреагируй на ответ пользователя (если он что-то спросил — коротко ответь); " + ask
        offer = ("предложи на выбор: объяснить любое правило команды S1–S6, проверить его код "
                 "(пусть пришлёт фрагмент) или задать вопросы на знание правил")
        if start:
            weak = find_rule(self.memory.facts.get("weak_spots", ""))
            about = f" (к нему относится правило {weak}: «{RULES[weak]}»)" if weak else ""
            return ("пользователь вернулся, о нём известно всё нужное: поприветствуй по имени, "
                    f"напомни про его слабое место{about} и " + offer)
        return "поблагодари за ответы, одной фразой подытожь, что узнал о пользователе, и " + offer

    def _say(self, x: dict, task: str) -> AIMessage:
        return self.chain.invoke({**x, "task": task})

    def _note_mistakes(self, rules: list[str]) -> None:
        mistakes = dict(self.memory.progress.get("mistakes", {}))
        for rule in rules:
            mistakes[rule] = mistakes.get(rule, 0) + 1
        self.memory.save_progress(mistakes=mistakes)

    # --- ветки режима наставника ---

    def _retrieve(self, x: dict) -> dict:
        """RAG: для вопроса или объяснения правила ищем фрагменты руководств.

        Для «другого» близость лучшего фрагмента решает, вопрос ли это про Go: выше порога — ветка «вопрос»
        с ответом по базе, ниже — разговор или не по теме, фрагменты не нужны.
        """
        if not self.kb or x["intent"] not in ("explain_rule", "other"):
            return {**x, "hits": []}
        rule = x["rule"]
        query = f"{rule}. {RULES[rule]} {x['input']}" if x["intent"] == "explain_rule" else x["input"]
        hits = self.kb.search(query, self.k)
        self.log("поиск: " + "; ".join(f"`{score:.2f}` {cite(doc)}" for doc, score in hits))
        intent, best = x["intent"], hits[0][1]
        if intent == "other":
            if best >= self.kb.threshold:
                intent = "question"
                self.log(f"близость {best:.2f} ≥ порога {self.kb.threshold} — вопрос по Go, отвечаем по базе")
            else:
                hits = []
                self.log(f"близость {best:.2f} < порога {self.kb.threshold} — не по базе знаний")
        return {**x, "intent": intent, "hits": hits}

    def _explain(self, x: dict) -> AIMessage:
        level, rule, hits = self.memory.level(), x["rule"], x.get("hits")
        self.log(f"ветка «объяснить»: {rule}, глубина по уровню `{level}`"
                 + (f", фрагментов из базы: {len(hits)}" if hits else ""))
        task = (f"объясни правило команды {rule}: «{RULES[rule]}». Уровень пользователя — {level}: "
                f"{DEPTH[level]}. Пример кода (до 10 строк) — из его проекта.")
        if not hits:
            return self._say(x, task)
        task += ("\nОпирайся на фрагменты руководств ниже (могут быть на английском), ссылайся на них номерами "
                 f"[1], [2]; список источников не пиши — его добавят:\n{format_context(hits)}")
        reply = strip_sources(self._say(x, task).content)
        return AIMessage(content=f"{reply}\n\n{format_sources(hits)}")

    def _answer(self, x: dict) -> AIMessage:
        level, hits = self.memory.level(), x["hits"]
        self.log(f"ветка «вопрос по базе знаний»: фрагментов {len(hits)}, уровень `{level}`")
        reply = self._say(x, "ответь на вопрос пользователя только по фрагментам руководств ниже (они могут быть "
                             f"на английском). Уровень пользователя — {level}: {DEPTH[level]}. Ссылайся на фрагменты "
                             "номерами [1], [2]; список источников не пиши — его добавят. Если в фрагментах ответа "
                             f"нет — так и скажи, не придумывай.\n\nФрагменты:\n{format_context(hits)}").content
        reply = strip_sources(reply)
        # номера в ответе пишет модель, а список источников — код: он совпадает с тем, что было в промпте
        return AIMessage(content=f"{reply}\n\n{format_sources(hits)}")

    def _review(self, x: dict) -> AIMessage:
        report = check_code.invoke({"code": x["code"]})  # инструмент: точный список нарушений
        rules = sorted({rule for _, rule, _ in lint_go(x["code"])})
        self.log(f"ветка «проверить код»: инструмент `check_code` → {', '.join(rules) or 'чисто'}")
        if rules:
            self._note_mistakes(rules)
            x["rule"] = rules[0]
        if not rules:
            return AIMessage(content="Инструмент check_code нарушений не нашёл — по правилам S1–S6 код чистый.")
        level = self.memory.level()
        reply = self._say(x, "пользователь прислал код. Инструмент check_code нашёл нарушения:\n"
                             f"{report}\nОбъясни каждое из них по одному предложению ({DEPTH[level]}) и покажи "
                             "исправленный код, где исправлены все они. Не называй нарушений, которых нет в списке.")
        # 3B-модель пересказывала отчёт не полностью (из S1, S4, S5 упомянула только S4) — список даём дословно
        return AIMessage(content=f"Инструмент check_code:\n{report}\n\n{reply.content.strip()}")

    def _quiz(self, x: dict) -> AIMessage:
        level, progress = self.memory.level(), self.memory.progress
        asked = progress.get("asked", [])
        candidates = [q for q in QUIZ if q["level"] == level and q["id"] not in asked]
        if not candidates:  # все вопросы уровня заданы — начинаем круг заново
            asked = [a for a in asked if not a.startswith(level[0])]
            candidates = [q for q in QUIZ if q["level"] == level]
        # сначала правила, на которых пользователь ошибался, и его слабое место
        weak, mistakes = find_rule(self.memory.facts.get("weak_spots", "")), progress.get("mistakes", {})
        q = max(candidates, key=lambda q: mistakes.get(q["rule"], 0) + (q["rule"] == weak))
        self.pending_quiz = q
        self.memory.save_progress(asked=asked + [q["id"]])
        self.log(f"ветка «квиз»: уровень `{level}`, вопрос `{q['id']}` (ответ {q['rule']})")
        return AIMessage(content=f"Вопрос уровня {level}. Какое правило (S1–S6) нарушено?\n```go\n{q['code']}\n```")

    def _grade(self, x: dict) -> AIMessage:
        q, self.pending_quiz = self.pending_quiz, None
        m = re.search(r"(?<!\w)[SsСс]?\s?([1-6])(?!\d)", x["input"])
        answer = f"S{m.group(1)}" if m else "не знаю"
        ok = answer == q["rule"]
        p = self.memory.progress
        level = self.memory.level()
        streak = max(p.get("streak", 0), 0) + 1 if ok else min(p.get("streak", 0), 0) - 1
        new_level = level
        if streak >= 2 and level != LEVELS[-1]:  # два верных подряд — вопросы сложнее
            new_level, streak = LEVELS[LEVELS.index(level) + 1], 0
        elif streak <= -2 and level != LEVELS[0]:  # две ошибки подряд — проще
            new_level, streak = LEVELS[LEVELS.index(level) - 1], 0
        mistakes = dict(p.get("mistakes", {}))
        if not ok:
            mistakes[q["rule"]] = mistakes.get(q["rule"], 0) + 1
        self.memory.save_progress(level=new_level, streak=streak, correct=p.get("correct", 0) + ok,
                                  wrong=p.get("wrong", 0) + (not ok), mistakes=mistakes)
        self.log(f"ветка «ответ на квиз»: ответ {answer}, верно {q['rule']} → {'верно' if ok else 'неверно'}"
                 + (f"; уровень `{level}` → `{new_level}`" if new_level != level else ""))
        self.last_rule = q["rule"]
        if not ok:
            self.quiz_context = {"q": q, "answer": answer}
        text = f"{'Верно' if ok else 'Неверно'}: здесь нарушено {q['rule']} — {q['why']}"
        if new_level != level:
            text += f" Дальше вопросы уровня {new_level}."
        return AIMessage(content=text)

    def _simplify(self, x: dict) -> AIMessage:
        rule = x["rule"]
        self.log(f"ветка «упростить»: {rule}, недоволен {self.frustration} ход(а) подряд")
        task = f"пользователь недоволен ({x['reason']}). Не оправдывайся. "
        if qc := self.quiz_context:
            task += (f"Он ответил на вопрос квиза {qc['answer']}, а верно {rule}. Код:\n{qc['q']['code']}\n"
                     f"Почему {rule}: {qc['q']['why']}\nПравило {rule}: «{RULES[rule]}». Объясни это ещё проще, "
                     f"своими словами, с бытовой аналогией; называй правило только как {rule}.")
        else:
            task += (f"Объясни правило {rule}: «{RULES[rule]}» совсем просто: одна бытовая аналогия и пример кода "
                     "3–6 строк «плохо / хорошо».")
        reply = self._say(x, task).content.strip()
        if self.frustration >= 2:  # просьбу в промпте 3B-модель пропускала — добавляем сами
            reply += "\n\nЕсли так и не стало понятнее — пришли свой фрагмент кода, разберём правило на нём."
        return AIMessage(content=reply)

    def _other(self, x: dict) -> AIMessage:
        self.log("ветка «другое»")
        return self._say(x, "если пользователь спрашивает о том, что говорил раньше, — ответь по истории и "
                            "краткому содержанию разговора, назови людей и детали оттуда. Если вопрос не про Go и не "
                            "про этот разговор — ты на него не отвечаешь и уточнений не просишь: одной фразой скажи, "
                            "что ты наставник только по код-стайлу Go, и предложи: объяснить правило, проверить код "
                            "или вопросы на знание правил.")

    # --- сеанс ---

    def start(self) -> str:
        """Первая реплика сеанса — без ввода пользователя."""
        self.log(f"сеанс начат: user_id=`{self.memory.user_id}`, pid процесса `{os.getpid()}`")
        self.log(f"не хватает фактов: {', '.join(self.memory.missing()) or 'нет'}")
        text = self.chain.invoke({"facts": format_facts(self.memory.facts), "task": self._task(start=True),
                                  "summary": "", "history": [], "input": "(пользователь открыл чат)"}
                                 ).content.strip()
        self.history.add_ai_message(text)
        self.last_question = text
        self.asked = next(iter(self.memory.missing()), "")
        return text

    def turn(self, user_text: str) -> str:
        config = {"configurable": {"session_id": self.memory.user_id}}
        if self.memory.missing() or self.asked:
            # знакомство: сначала факты — следующий вопрос должен учитывать только что сказанное
            self.memory.save_context(
                {"input": user_text, "question": self.last_question, "field": self.asked}, {})
            missing = self.memory.missing()
            self.log(f"дальше: {'спросить ' + missing[0] if missing else 'режим наставника — все факты собраны'}")
            variables = self.memory.load_memory_variables({})
            text = self.chat_chain.invoke(
                {"input": user_text, "facts": variables["facts"], "summary": self.history.prompt_summary(),
                 "task": self._task(start=False)}, config=config).content.strip()
            self.asked = next(iter(missing), "")
        else:
            self.log(f"история: в промпт {len(self.history.messages)} из {len(self.history.full)} сообщений"
                     + (", плюс резюме" if self.history.prompt_summary() else ""))
            reply = self.mentor_chain.invoke(
                {"input": user_text, "facts": format_facts(self.memory.facts),
                 "summary": self.history.prompt_summary()}, config=config)
            text = reply.content.strip()
            self.last_user = user_text
        self.last_question = text
        return text

    def chat(self, script: list[str] | None = None):
        """REPL. script — заранее заданные реплики (для демонстрации); без него читает input()."""
        print(f"ассистент> {self.start()}\n")
        replies = iter(script) if script is not None else None
        while True:
            if replies is None:
                user_text = input("вы> ").strip()
            else:
                user_text = next(replies, "/exit")
                print(f"вы> {user_text}")
            if user_text in ("/exit", "/quit", "выход"):
                self.log("сеанс завершён командой выхода")
                break
            if user_text:
                print(f"ассистент> {self.turn(user_text)}\n")


Writing fact_agent.py


## База знаний

Сколько чанков дал каждый источник и как выглядит чанк.

In [4]:
from collections import Counter
from pathlib import Path

import kb

chunks = kb.load_chunks(KB_DIR)
sizes = Counter(c.metadata["source"] for c in chunks)
print(f"{'источник':28} {'чанков':>6} {'символов в среднем':>18}")
for source, n in sizes.items():
    avg = sum(len(c.page_content) for c in chunks if c.metadata["source"] == source) // n
    print(f"{source:28} {n:6} {avg:18}")
print(f"{'всего':28} {len(chunks):6}")

example = next(c for c in chunks if c.metadata["section"] == "Receiver Names")
print("\n", example.metadata, "\n", example.page_content, sep="")

источник                     чанков символов в среднем
code_review_comments.md          41                634
effective_go.md                 119                864
uber_style.md                   100                802
team_style.md                     7                606
всего                           267

{'source': 'code_review_comments.md', 'title': 'Go Code Review Comments', 'section': 'Receiver Names', 'url': 'https://go.dev/wiki/CodeReviewComments'}
Go Code Review Comments. Receiver Names
The name of a method's receiver should be a reflection of its identity; often a one or two letter abbreviation of its type suffices (such as "c" or "cl" for "Client"). Don't use generic names such as "me", "this" or "self", identifiers typical of object-oriented languages that gives the method a special meaning. In Go, the receiver of a method is just another parameter and therefore, should be named accordingly. The name need not be as descriptive as that of a method argument, as its role is

### Индекс

Эмбеддинги всех чанков — один раз; индекс сохраняется в `kb_index.json`.

In [5]:
Path(INDEX_FILE).unlink(missing_ok=True)  # чистый старт — только для демонстрации
base = kb.KnowledgeBase.build_or_load(KB_DIR, INDEX_FILE)

индекс построен: 267 чанков за 11 с, сохранён в `kb_index.json`


## Качество поиска

14 вопросов по-русски, для каждого заранее известно, в каких разделах ответ (`kb.EVAL`). Ранг — место первого верного раздела среди 4 найденных.

In [6]:
rows = kb.evaluate(base, k=4)
print(f"{'ранг':>4}  {'близость':>8}  вопрос → лучший найденный раздел")
for r in rows:
    print(f"{r['rank'] or '—':>4}  {r['score']:8.3f}  {r['question']}\n{'':16}→ {r['top1']}")
hit1 = sum(r["rank"] == 1 for r in rows) / len(rows)
hit4 = sum(r["rank"] is not None for r in rows) / len(rows)
mrr = sum(1 / r["rank"] for r in rows if r["rank"]) / len(rows)
print(f"\nhit@1 = {hit1:.2f}   hit@4 = {hit4:.2f}   MRR = {mrr:.2f}")

ранг  близость  вопрос → лучший найденный раздел
   1     0.560  Как называть ресивер метода?
                → Go Code Review Comments — Receiver Names
   1     0.604  Когда метод должен принимать указатель, а когда значение?
                → Go Code Review Comments — Receiver Type
   1     0.584  С какой буквы начинать текст ошибки и ставить ли точку в конце?
                → Go Code Review Comments — Error Strings
   1     0.633  Как добавить контекст к ошибке, чтобы errors.Is продолжал работать?
                → Uber Go Style Guide — Guidelines > Errors > Error Wrapping
   1     0.667  Можно ли использовать panic для обработки обычных ошибок?
                → Код-стайл команды — S3. Без panic в HTTP-обработчиках
   1     0.565  Где объявлять интерфейс: рядом с реализацией или там, где он используется?
                → Go Code Review Comments — Interfaces
   1     0.660  Как писать аббревиатуры вроде URL и ID в именах?
                → Go Code Review Comments — Initialisms
   

### Порог «вопрос по Go / не по теме»

Близость лучшего фрагмента для вопросов из `EVAL` и для вопросов не по теме; порог — посередине. Он сохраняется рядом с индексом.

In [7]:
cal = base.calibrate()
for name, scores in (("по теме", cal["inside"]), ("не по теме", cal["outside"])):
    print(f"{name}: min {min(scores.values()):.3f}, max {max(scores.values()):.3f}")
print("не по теме:", {q: round(s, 3) for q, s in cal["outside"].items()})
print("порог:", cal["threshold"])

по теме: min 0.534, max 0.677
не по теме: min 0.309, max 0.452
не по теме: {'Какая сегодня погода?': 0.452, 'Посоветуй рецепт борща.': 0.445, 'Кто выиграл чемпионат мира по футболу?': 0.309, 'Сколько стоит биткоин?': 0.449}
порог: 0.493


## Модель без базы и с базой

Одни и те же вопросы: голая модель (`PLAIN_PROMPT | llm`) и RAG-цепочка LCEL (поиск → фрагменты в промпт → ответ → источники).

In [8]:
import fact_agent as fa

llm = fa.make_llm(0.3)
plain, rag = kb.make_plain_chain(llm), kb.make_rag_chain(base, llm)
for question in [
    'Как называть ресивер метода?',
    'С какой буквы начинать текст ошибки и ставить ли точку в конце?',
    'Где объявлять интерфейс: рядом с реализацией или там, где он используется?',
]:
    print("=" * 100, "\nВОПРОС:", question)
    print("\n--- без базы:\n" + plain.invoke({"question": question}).strip())
    print("\n--- с базой (RAG):\n" + rag.invoke({"question": question}))

ВОПРОС: Как называть ресивер метода?



--- без базы:
Ресивер метода называют сигнатурным аргументом.



--- с базой (RAG):
Название ресивера должно быть коротким, обычно одной или двух букв, соответствующими типу (например, "c" или "cl" для "Client"). Не используйте общепринятые в объектно-ориентированном программировании имена, такие как "me", "this" или "self".

Источники:
[1] Go Code Review Comments — Receiver Names (https://go.dev/wiki/CodeReviewComments)
[2] Uber Go Style Guide — Guidelines > Receivers and Interfaces (https://github.com/uber-go/guide/blob/master/style.md)
[3] Effective Go — Methods > Pointers vs. Values (https://go.dev/doc/effective_go)
[4] Go Code Review Comments — Receiver Type (https://go.dev/wiki/CodeReviewComments)
ВОПРОС: С какой буквы начинать текст ошибки и ставить ли точку в конце?



--- без базы:
Текст ошибок начинается с маленькой буквы и заканчивается точкой.



--- с базой (RAG):
Ошибка текста должна начинаться со строчной буквы и не должна заканчиваться на запятую или точку. [1]

Источники:
[1] Go Code Review Comments — Error Strings (https://go.dev/wiki/CodeReviewComments)
[2] Uber Go Style Guide — Guidelines > Errors > Error Types (https://github.com/uber-go/guide/blob/master/style.md)
[3] Uber Go Style Guide — Guidelines > Errors > Error Wrapping (https://github.com/uber-go/guide/blob/master/style.md)
ВОПРОС: Где объявлять интерфейс: рядом с реализацией или там, где он используется?



--- без базы:
Интерфейс объявляют рядом с его реализацией, чтобы было ясно, что этот интерфейс реализован этим кодом.



--- с базой (RAG):
Интерфейс лучше объявлять рядом с тем пакетом, где он используется. Это позволяет избежать необходимости возвращать указатели на интерфейс, что упрощает тестирование и позволяет добавлять новые методы без значительных изменений в коде.

Источники:
[1] Go Code Review Comments — Interfaces (https://go.dev/wiki/CodeReviewComments)
[2] Uber Go Style Guide — Guidelines > Pointers to Interfaces (https://github.com/uber-go/guide/blob/master/style.md)
[3] Effective Go — The blank identifier > Interface checks (https://go.dev/doc/effective_go)
[4] Effective Go — Interfaces and other types > Interfaces (https://go.dev/doc/effective_go)


## Перезапуск ядра

**Kernel → Restart.** Индекс и порог — на диске: после перезапуска эмбеддинги документов не пересчитываются.

In [1]:
import os

# чат-модель и модель эмбеддингов — два OpenAI-совместимых сервера (llama-server локально или на Kaggle)
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # Qwen2.5-3B-Instruct
os.environ.setdefault("LLM_MODEL", "local")
os.environ.setdefault("EMB_BASE_URL", "http://127.0.0.1:8081/v1")  # bge-m3, llama-server --embeddings
os.environ.setdefault("EMB_MODEL", "bge-m3")

KB_DIR = "kb"                # руководства по стилю Go (Markdown)
INDEX_FILE = "kb_index.json"  # векторный индекс на диске
MEMORY_FILE = "memory.json"
USER_ID = "artur"

import kb
import fact_agent as fa

base = kb.KnowledgeBase.build_or_load(KB_DIR, INDEX_FILE)

индекс прочитан из `kb_index.json`: 267 чанков, эмбеддинги не пересчитывались; порог 0.493


## Наставник с RAG

Профиль `artur` из лаб 2–3 (факты и прогресс — в `memory.json`). Шаг поиска виден в служебном логе: найденные разделы с близостью и решение по порогу. Вопрос про ресивер и про `context.Context` в структуре идут в ветку «вопрос по базе знаний», «Объясни правило S5» — в «объяснить» с фрагментами, погода — ниже порога, квиз работает как раньше.

In [2]:
log = fa.LogCell("Служебный лог — наставник с RAG")

**Служебный лог — наставник с RAG**

- `12:24:13` прочитан `memory.json`, профиль `artur`: `{"name": "Артур", "go_level": "junior", "go_experience": "более года", "stack": "PostgreSQL, Docker, NixOS", "project": "сервис заказов с REST API", "weak_spots": "управление ошибками в Go", "goal": "писать идиоматичный Go и проходить ревью с первого раза"}`
- `12:24:13` прогресс: `{"mistakes": {"S1": 1, "S4": 1, "S5": 2}, "asked": ["j1", "j2", "m2"], "level": "middle", "streak": -1, "correct": 2, "wrong": 1}`
- `12:24:13` память диалога: `summary`, окно 6 сообщений; база знаний: 267 чанков, порог 0.493
- `12:24:13` сеанс начат: user_id=`artur`, pid процесса `126`
- `12:24:13` не хватает фактов: нет
- `12:24:14` история: в промпт 0 из 1 сообщений
- `12:24:15` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`other`, rule=`S1`
- `12:24:15` поиск: `0.59` Go Code Review Comments — Receiver Names; `0.57` Effective Go — Interfaces and other types > Interfaces and methods; `0.55` Effective Go — Names > Interface names; `0.55` Effective Go — Methods > Pointers vs. Values
- `12:24:15` близость 0.59 ≥ порога 0.493 — вопрос по Go, отвечаем по базе
- `12:24:15` ветка «вопрос по базе знаний»: фрагментов 4, уровень `middle`
- `12:24:16` история: в промпт 2 из 3 сообщений
- `12:24:16` классификатор: `{"intent":"review_code","rule":"S1"}` → парсер: intent=`other`, rule=`S1`
- `12:24:16` поиск: `0.51` Effective Go — Methods > Pointers vs. Values; `0.50` Go Code Review Comments — Receiver Names; `0.50` Effective Go — Names > Getters; `0.50` Effective Go — Embedding
- `12:24:16` близость 0.51 ≥ порога 0.493 — вопрос по Go, отвечаем по базе
- `12:24:16` ветка «вопрос по базе знаний»: фрагментов 4, уровень `middle`
- `12:24:19` история: в промпт 4 из 5 сообщений
- `12:24:19` классификатор: `{"intent":"explain_rule","rule":"S5"}` → парсер: intent=`explain_rule`, rule=`S5`
- `12:24:19` поиск: `0.76` Код-стайл команды — S5. Оборачивать ошибки; `0.63` Код-стайл команды — S2. Ошибки не игнорировать; `0.60` Uber Go Style Guide — Guidelines > Errors > Error Wrapping; `0.57` Uber Go Style Guide — Guidelines > Errors > Error Wrapping
- `12:24:19` ветка «объяснить»: S5, глубина по уровню `middle`, фрагментов из базы: 4
- `12:24:24` история: в промпт 6 из 7 сообщений
- `12:24:25` классификатор: `{"intent":"explain_rule","rule":"S4"}` → парсер: intent=`other`, rule=`S4`
- `12:24:25` поиск: `0.59` Код-стайл команды — S4. context.Context первым параметром; `0.52` Go Code Review Comments — Contexts; `0.52` Go Code Review Comments — Contexts; `0.48` Uber Go Style Guide — Guidelines > Avoid Embedding Types in Public Structs
- `12:24:25` близость 0.59 ≥ порога 0.493 — вопрос по Go, отвечаем по базе
- `12:24:25` ветка «вопрос по базе знаний»: фрагментов 4, уровень `middle`
- `12:24:31` резюме: свёрнуто сообщений 2 → «Пользователь: Подскажи, какое имя дать получателю метода у типа Client? Наставник: Для типа Client получатель метода может быть "c" или "cl" (Client), указывая его роль.»
- `12:24:31` история: в промпт 6 из 9 сообщений, плюс резюме
- `12:24:31` классификатор: `{"intent":"review_code","rule":"S6"}` → парсер: intent=`other`, rule=`S6`
- `12:24:31` поиск: `0.45` Uber Go Style Guide — Guidelines > Use `"time"` to handle time; `0.43` Uber Go Style Guide — Guidelines > Use `"time"` to handle time > Use `time.Duration` for periods of time; `0.42` Uber Go Style Guide — Guidelines > Avoid Mutable Globals; `0.42` Uber Go Style Guide — Guidelines > Use `"time"` to handle time > Use `time.Time` for instants of time
- `12:24:31` близость 0.45 < порога 0.493 — не по базе знаний
- `12:24:31` ветка «другое»
- `12:24:33` резюме: свёрнуто сообщений 2 → «Пользователь: А можно назвать его self, как в Python? Наставник: Название "self" не рекомендуется в Go, так как это может вызвать путаницу с методами объектов в объектно-ориентированных языках.»
- `12:24:33` история: в промпт 6 из 11 сообщений, плюс резюме
- `12:24:33` классификатор: `{"intent":"quiz","rule":"S1"}` → парсер: intent=`quiz`, rule=`S1`
- `12:24:33` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 2}, "asked": ["j1", "j2", "m2", "m1"], "level": "middle", "streak": -1, "correct": 2, "wrong": 1}`
- `12:24:33` ветка «квиз»: уровень `middle`, вопрос `m1` (ответ S4)
- `12:24:35` резюме: свёрнуто сообщений 2 → «Пользователь: Объясни правило S5.
Наставник: Правило S5 из командного код-стайла Go гласит: "Ошибки, возвращаемые наверх, оборачиваются: `fmt.Errorf("контекст: %w", err)`. Глагол — `%w`, а не...".»
- `12:24:35` история: в промпт 6 из 13 сообщений, плюс резюме
- `12:24:35` классификатор: `{"intent":"explain_rule","rule":"S4"}` → парсер: intent=`quiz_answer`, rule=`S4`
- `12:24:35` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 2}, "asked": ["j1", "j2", "m2", "m1"], "level": "middle", "streak": 1, "correct": 3, "wrong": 1}`
- `12:24:35` ветка «ответ на квиз»: ответ S4, верно S4 → верно
- `12:24:36` резюме: свёрнуто сообщений 2 → «Пользователь: Можно ли хранить context.Context в поле структуры? Наставник: Хранить `context.Context` в поле структуры не рекомендуется. Это делает структуру более сложной и затрудняет управление контекстом.»
- `12:24:36` сеанс завершён командой выхода

In [3]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log, kb=base)
agent.chat(script=[
    'Подскажи, какое имя дать получателю метода у типа Client?',
    'А можно назвать его self, как в Python?',
    'Объясни правило S5.',
    'Можно ли хранить context.Context в поле структуры?',
    'Какая сегодня погода?',
    'Задай мне вопрос на знание правил.',
    'S4',
    '/exit',
])

ассистент> Привет, Артур! Поехали, давай ещё раз проверим правило S5: ошибки, возвращаемые наверх, оборачиваются: fmt.Errorf("контекст: %w", err).

вы> Подскажи, какое имя дать получателю метода у типа Client?


ассистент> Для типа Client получатель метода должен быть коротким и отражать его роль. Например, можно использовать "c" или "cl" (Client). Такое имя указывает на тип объекта, без добавления специального значения.

Источники:
[1] Go Code Review Comments — Receiver Names (https://go.dev/wiki/CodeReviewComments)
[2] Effective Go — Interfaces and other types > Interfaces and methods (https://go.dev/doc/effective_go)
[3] Effective Go — Names > Interface names (https://go.dev/doc/effective_go)
[4] Effective Go — Methods > Pointers vs. Values (https://go.dev/doc/effective_go)

вы> А можно назвать его self, как в Python?


ассистент> Название "self" не рекомендуется в Go, так как это может вызвать путаницу с методами объектов в объектно-ориентированных языках. В Go получатель метода должен быть коротким и отражать тип объекта. Например, "c" или "cl" для типа Client.

Источники:
[1] Effective Go — Methods > Pointers vs. Values (https://go.dev/doc/effective_go)
[2] Go Code Review Comments — Receiver Names (https://go.dev/wiki/CodeReviewComments)
[3] Effective Go — Names > Getters (https://go.dev/doc/effective_go)
[4] Effective Go — Embedding (https://go.dev/doc/effective_go)

вы> Объясни правило S5.


ассистент> Правило S5 из командного код-стайла Go гласит:

- Ошибки, возвращаемые наверх, оборачиваются: `fmt.Errorf("контекст: %w", err)`.
- Глагол — `%w`, а не `%v`.
- Контекст — короткий, без «failed to» и «ошибка при».
- Например:
  ```go
  // Плохо
  if err := row.Scan(&u.Name); err != nil {
    return nil, err
  }

  // Хорошо
  if err := row.Scan(&u.Name); err != nil {
    return nil, fmt.Errorf("scan user %d: %w", id, err)
  }
  ```

Это правило важно для следующих причин:
1. **Контекст**: `%w` позволяет получить информацию о причине ошибки, что полезно для отладки и анализа.
2. **Короткий контекст**: Короткие сообщения об ошибках легче воспринимаются и анализируются.
3. **Использование `%w` вместо `%v`**: `%w` предоставляет больше информации о типе ошибки, что полезно для проверки и отладки.

Примеры:
- **Плохо**:
  ```go
  if err := row.Scan(&u.Name); err != nil {
    return nil, err
  }
  ```

- **Хорошо**:
  ```go
  if err := row.Scan(&u.Name); err != nil {
    return nil, 

ассистент> Хранить `context.Context` в поле структуры не рекомендуется. Это делает структуру более сложной и затрудняет управление контекстом.

Правило из командного код-стайл Go гласит:

- **Не храните контекст в полях структур**: Контекст должен передаваться явно по параметрам методов.
- **Добавьте контекстный параметр в каждый метод**: Если метод зависит от контекста, он должен принимать `ctx context.Context` в качестве первого параметра.
- **Контекст не храните в полях структур**: Это делает структуру более сложной и затрудняет управление контекстом.

Примеры:

- **Плохо**:
  ```go
  type Client struct {
    ctx context.Context
  }

  func (c *Client) DoSomething() error {
    // ...
    return c.ctx.Err()
  }
  ```

- **Хорошо**:
  ```go
  type Client struct {
  }

  func (c *Client) DoSomething(ctx context.Context) error {
    // ...
    return ctx.Err()
  }
  ```

Использование контекста в качестве параметра методов делает код более чистым, легким для управления и понимания.

Ис

ассистент> К сожалению, я не могу получить актуальную погоду, так как у меня нет доступа к внешним источникам данных. Но ты можешь легко найти погоду на специализированных сайтах или приложениях.

вы> Задай мне вопрос на знание правил.


ассистент> Вопрос уровня middle. Какое правило (S1–S6) нарушено?
```go
// FindUser ищет пользователя по id.
func (r *Repo) FindUser(id int64) (*User, error) {
    row := r.db.QueryRow("SELECT name FROM users WHERE id = $1", id)
```

вы> S4


ассистент> Верно: здесь нарушено S4 — `FindUser` ходит в БД, но не принимает контекст: нужно `FindUser(ctx context.Context, id int64)` и `QueryRowContext(ctx, ...)`.

вы> /exit


## Свой диалог

```python
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=fa.LogCell(), kb=base)
agent.chat()   # /exit — выход
```